# Survival Progressive Missingness Analysis

Notebook for discrete-time survival progressive missingness analysis. It mirrors the classification progressive missingness workflow, but builds replicate-level C-index values from stored survival risk scores.

Expected prediction columns include `event_time`, `event_observed`, and either retained `inner_model_<k>_risk` columns or an ensemble risk computed as the mean of the retained inner-model risks.


In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'analysis' else cwd

DATASET_NAME = 'mmColorectal'
LABEL_NAME = 'OS'
DISPLAY_DATASET_NAME = DATASET_NAME

TRAIN_DEGRADING_MODALITY = 'GLOBAL'
RETRAIN_OUTER = False
USE_ENSEMBLE = False
DISTILLATION_MODEL_NAMES = []  # _KD methods are detected automatically

N_BOOTSTRAP = 2000
BOOTSTRAP_CONFIDENCE = 0.95
BOOTSTRAP_RANDOM_SEED = 42

RESULTS_TAG = f'{DATASET_NAME}_{LABEL_NAME}' if str(LABEL_NAME).strip() else DATASET_NAME
RESULTS_ROOT = PROJECT_ROOT / 'results' / RESULTS_TAG / 'training_runs'
if not RESULTS_ROOT.exists():
    raise FileNotFoundError(f'Results root does not exist: {RESULTS_ROOT}')
if not RESULTS_ROOT.is_dir():
    raise NotADirectoryError(f'Expected a directory at: {RESULTS_ROOT}')

RETRAIN_TAG = f"retrain{str(bool(RETRAIN_OUTER)).lower()}"
PREDICTION_TAG = 'ensemble' if bool(USE_ENSEMBLE) else 'inner_models'
OUTPUT_DIR = (
    (cwd / 'survival_progressive_missingness_analysis_outputs') if cwd.name == 'analysis' else (cwd / 'analysis' / 'survival_progressive_missingness_analysis_outputs')
) / RESULTS_TAG / TRAIN_DEGRADING_MODALITY.lower() / RETRAIN_TAG / PREDICTION_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = OUTPUT_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINT_NAME = LABEL_NAME if str(LABEL_NAME).strip() else RESULTS_ROOT.parent.name.replace(f'{DATASET_NAME}_', '', 1)
RESULTS_ROOT


In [ ]:
import sys
import importlib
import pandas as pd

ANALYSIS_DIR = cwd if cwd.name == 'analysis' else (cwd / 'analysis')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

import results_analysis as ra
ra = importlib.reload(ra)

HAVE_MPL = ra.HAVE_MPL
resolve_requested_model_names = ra.resolve_requested_model_names
load_all_survival_test_predictions = ra.load_all_survival_test_predictions
expand_inner_model_survival_predictions = ra.expand_inner_model_survival_predictions
aggregate_member_patient_survival_predictions = ra.aggregate_member_patient_survival_predictions
build_replicate_cindex_table = ra.build_replicate_cindex_table
cindex_replicates_to_auc_compatible = ra.cindex_replicates_to_auc_compatible
build_level1_summary = ra.build_level1_summary
build_method_plot_summary = ra.build_method_plot_summary
build_method_level_metrics = ra.build_method_level_metrics
build_degradation_curve_summary = ra.build_degradation_curve_summary
build_metric_ordering_table = ra.build_metric_ordering_table
compute_level1_global_friedman = ra.compute_level1_global_friedman
compute_level2_pairwise_tests = ra.compute_level2_pairwise_tests
select_level2_plot_pairs = ra.select_level2_plot_pairs
build_top_equivalent_group_counts = ra.build_top_equivalent_group_counts
build_general_results_summary = ra.build_general_results_summary
rename_auc_outputs_for_cindex = ra.rename_auc_outputs_for_cindex
rename_general_results_summary_for_cindex = ra.rename_general_results_summary_for_cindex
plot_level1_auc_heatmaps = ra.plot_level1_auc_heatmaps
plot_method_line_triplet = ra.plot_method_line_triplet
plot_level2_significant_pairs_heatmap = ra.plot_level2_significant_pairs_heatmap
plot_level3_pairwise_condition_matrices = ra.plot_level3_pairwise_condition_matrices


## Build C-index summaries

The notebook computes replicate-level C-index values and then reuses the same AUPMC, degradation, ranking, and condition-level logic used for AUC. Internally, C-index is mapped to an `auc`-compatible column only while calling the shared functions; saved CSVs use C-index names.


In [ ]:
REQUESTED_MODEL_NAMES = resolve_requested_model_names(
    results_root=RESULTS_ROOT,
    dataset_name=DATASET_NAME,
    train_degrading_modality=TRAIN_DEGRADING_MODALITY,
    retrain_outer=RETRAIN_OUTER,
)
if not REQUESTED_MODEL_NAMES:
    raise ValueError('No model folders were detected for the requested dataset / location / retrain flag.')

raw_predictions_df, missing_prediction_files = load_all_survival_test_predictions(
    results_root=RESULTS_ROOT,
    dataset_name=DATASET_NAME,
    train_degrading_modality=TRAIN_DEGRADING_MODALITY,
    model_names=REQUESTED_MODEL_NAMES,
    retrain_outer=RETRAIN_OUTER,
    use_ensemble=USE_ENSEMBLE,
)
if raw_predictions_df.empty:
    raise ValueError('No survival test_predictions.csv files with usable risk scores were found.')

member_prediction_df = expand_inner_model_survival_predictions(raw_predictions_df, use_ensemble=USE_ENSEMBLE)
member_patient_df = aggregate_member_patient_survival_predictions(member_prediction_df)
replicate_cindex_df = build_replicate_cindex_table(member_patient_df)
replicate_metric_df = cindex_replicates_to_auc_compatible(replicate_cindex_df)

level1_summary_internal_df = build_level1_summary(replicate_metric_df)
level1_global_friedman_df = compute_level1_global_friedman(level1_summary_internal_df)
method_plot_summary_internal_df = build_method_plot_summary(
    replicate_auc_df=replicate_metric_df,
    n_bootstrap=N_BOOTSTRAP,
    confidence=BOOTSTRAP_CONFIDENCE,
    random_seed=BOOTSTRAP_RANDOM_SEED,
)
degradation_curve_summary_internal_df = build_degradation_curve_summary(
    method_plot_summary_df=method_plot_summary_internal_df,
    distillation_model_names=DISTILLATION_MODEL_NAMES,
)
method_level_metrics_internal_df, best_fixed_train_curve_internal_df = build_method_level_metrics(
    replicate_auc_df=replicate_metric_df,
    level1_df=level1_summary_internal_df,
    distillation_model_names=DISTILLATION_MODEL_NAMES,
    n_bootstrap=N_BOOTSTRAP,
    confidence=BOOTSTRAP_CONFIDENCE,
    random_seed=BOOTSTRAP_RANDOM_SEED,
)
method_metric_ordering_internal_df = build_metric_ordering_table(
    method_level_metrics_internal_df,
    distillation_model_names=DISTILLATION_MODEL_NAMES,
)

level1_summary_df = rename_auc_outputs_for_cindex(level1_summary_internal_df)
method_plot_summary_df = rename_auc_outputs_for_cindex(method_plot_summary_internal_df)
degradation_curve_summary_df = rename_auc_outputs_for_cindex(degradation_curve_summary_internal_df)
method_level_metrics_df = rename_auc_outputs_for_cindex(method_level_metrics_internal_df)
method_metric_ordering_df = rename_auc_outputs_for_cindex(method_metric_ordering_internal_df)
best_fixed_train_curve_df = rename_auc_outputs_for_cindex(best_fixed_train_curve_internal_df)

paths = {
    'replicate_cindex_table.csv': replicate_cindex_df,
    'method_condition_mean_cindex_summary.csv': level1_summary_df,
    'level1_global_friedman.csv': level1_global_friedman_df,
    'method_plot_summary.csv': method_plot_summary_df,
    'degradation_curve_summary.csv': degradation_curve_summary_df,
    'method_level_metrics.csv': method_level_metrics_df,
    'method_metric_orderings.csv': method_metric_ordering_df,
    'best_fixed_train_curve.csv': best_fixed_train_curve_df,
}
for file_name, df in paths.items():
    path = OUTPUT_DIR / file_name
    df.to_csv(path, index=False)
    print('Saved:', path)

replicate_unit_label = 'seed x outer_fold ensemble' if bool(USE_ENSEMBLE) else ('seed x outer_fold' if bool(RETRAIN_OUTER) else 'seed x outer_fold x inner_model_k')
print(f'Replicate unit: {replicate_unit_label}')
print('Missing prediction files:', len(missing_prediction_files))


## Plots and method-level tables

Plots use the same visual functions as the AUC workflow, with C-index passed through the internal `mean_auc` compatibility column. Saved tables use C-index terminology.


In [ ]:
print('Global Friedman test across heatmap cells:')
display(level1_global_friedman_df)

level1_model_count_tag = f"{level1_summary_internal_df['model_name'].nunique()}models" if not level1_summary_internal_df.empty else '0models'
if HAVE_MPL:
    plot_level1_auc_heatmaps(
        summary_df=method_plot_summary_internal_df,
        friedman_global_df=level1_global_friedman_df,
        title=f'C-index ± 95% CI Heatmaps | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME} | degrading_modality = {TRAIN_DEGRADING_MODALITY}',
        figures_dir=FIGURES_DIR,
        file_name=f'level1_cindex_{level1_model_count_tag}.png',
    )
    plot_method_line_triplet(
        summary_df=method_plot_summary_internal_df,
        metric_col='mean_auc',
        title=f'Mean C-index Curves | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME} | degrading_modality = {TRAIN_DEGRADING_MODALITY}',
        ylabel='Mean C-index',
        figures_dir=FIGURES_DIR,
        file_name=f'method_level_mean_cindex_curves_{RESULTS_TAG}_{TRAIN_DEGRADING_MODALITY.lower()}.png',
        panel_titles={
            'train': 'Train-time missingness',
            'test': 'Test-time missingness',
            'envelope': 'Best fixed train-missingness',
        },
        y_limits=None,
        clip_ci=False,
    )
    plot_method_line_triplet(
        summary_df=degradation_curve_summary_internal_df,
        metric_col='degradation_ratio',
        title=f'C-index Degradation Curves | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME} | degrading_modality = {TRAIN_DEGRADING_MODALITY}',
        ylabel='baseline C-index / C-index',
        figures_dir=FIGURES_DIR,
        file_name=f'method_level_cindex_degradation_curves_{RESULTS_TAG}_{TRAIN_DEGRADING_MODALITY.lower()}.png',
        ci_col='degradation_ratio_ci95',
        panel_titles={
            'train': 'Train-time missingness',
            'test': 'Test-time missingness',
            'envelope': 'Best fixed train-missingness',
        },
        y_limits=None,
        clip_ci=False,
    )
else:
    print('Matplotlib not available; skipping plots.')

display(method_level_metrics_df)
display(method_metric_ordering_df)


## Condition-level comparisons

Pairwise Wilcoxon signed-rank tests are performed within each train/test missingness condition using replicate-level C-index values. FDR correction is applied inside each condition.


In [ ]:
level2_pairwise_internal_df = compute_level2_pairwise_tests(
    level1_df=level1_summary_internal_df,
    replicate_auc_df=replicate_metric_df,
)
level2_plot_internal_df = select_level2_plot_pairs(level2_pairwise_internal_df)

level2_significant_pairs_df = (
    level2_pairwise_internal_df.loc[level2_pairwise_internal_df['significant_fdr_0p05']]
    .sort_values(['winner_model', 'loser_model', 'train_missing_prop', 'test_missing_prop'])
    .reset_index(drop=True)
)
level2_significant_pairs_df = rename_auc_outputs_for_cindex(level2_significant_pairs_df)
level2_plot_df = rename_auc_outputs_for_cindex(level2_plot_internal_df)
top_equivalent_group_counts_df = build_top_equivalent_group_counts(level2_plot_internal_df)
general_results_summary_df = rename_general_results_summary_for_cindex(
    build_general_results_summary(
        method_level_metrics_df=method_level_metrics_internal_df,
        level2_plot_df=level2_plot_internal_df,
    )
)

paths = {
    'wilcoxon_significant.csv': level2_significant_pairs_df,
    'top_equivalent_group_counts.csv': top_equivalent_group_counts_df,
    'general_results_summary.csv': general_results_summary_df,
}
for file_name, df in paths.items():
    path = OUTPUT_DIR / file_name
    df.to_csv(path, index=False)
    print('Saved:', path)

root_tag = RESULTS_ROOT.parent.name
location_tag = TRAIN_DEGRADING_MODALITY.lower()
model_count_tag = f"{level1_summary_internal_df['model_name'].nunique()}models" if not level1_summary_internal_df.empty else '0models'
print('Significant pairwise comparisons:', len(level2_significant_pairs_df))
print('Condition-summary heatmap cells:', len(level2_plot_df))

if HAVE_MPL:
    plot_level3_pairwise_condition_matrices(
        level2_pairwise_df=level2_pairwise_internal_df,
        title=f'Condition-level Pairwise Significant ΔC-index Matrices | ordered by within-condition mean C-index | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME}',
        figures_dir=FIGURES_DIR,
        file_name=f'level2_pairwise_condition_matrices_cindex_{root_tag}_{location_tag}_{model_count_tag}.png',
    )
    plot_level2_significant_pairs_heatmap(
        level2_plot_df=level2_plot_internal_df,
        title=f'Condition-level Top Equivalent Group vs First Significantly Lower-Ranked Method | C-index | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME}',
        figures_dir=FIGURES_DIR,
        file_name=f'level3_significant_pairs_cindex_{root_tag}_{location_tag}_{model_count_tag}.png',
    )
else:
    print('Matplotlib not available; skipping condition-level plots.')


## General results summary

Scenario-level winners and flexibility counts using C-index-based method-level metrics and condition-level top-equivalent groups.


In [ ]:
display(general_results_summary_df)
display(top_equivalent_group_counts_df)
